In [1]:
import pandas as pd
import json
import os
import subprocess
import sys

current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()

root_dir = os.path.abspath(os.path.join(current_dir, ".."))

GOLD_DATA_PATH = os.path.join(root_dir, 'data','sample', 'lab4_gold_ie.jsonl')
AUDIT_REPORT_PATH = os.path.join(root_dir, 'docs', 'audit_summary_lab4.md')

print(f"Шукаю дані в: {root_dir}")

try:
    import tabulate
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tabulate"])

sys.path.insert(0, os.path.join(root_dir, "src"))

import importlib
import ie_rules
importlib.reload(ie_rules)
from ie_rules import extract_all

print("\n--- Оцінка на Gold Standard ---")
if os.path.exists(GOLD_DATA_PATH):
    with open(GOLD_DATA_PATH, 'r', encoding='utf-8') as f:
        gold = [json.loads(line) for line in f]
    eval_res = []
    for item in gold:
        ext = extract_all(item['text'])
        match = any(e['field_type'] == item['field_type'] and abs(e['start_char'] - item['start_char']) <= 3 for e in ext)
        eval_res.append({"type": item['field_type'], "correct": match})
    report = pd.DataFrame(eval_res).groupby('type')['correct'].mean().reset_index()
    report.columns = ['Field Type', 'Precision']
    print(report.to_string(index=False))
else:
    print(f"ПОМИЛКА: Файл не знайдено за шляхом: {GOLD_DATA_PATH}")
    report = pd.DataFrame([["DATE", 0.0], ["AMOUNT", 0.0], ["PHONE", 0.0]], columns=['Field Type', 'Precision'])

print("\n--- Крок 3: Пошук реальних помилок (False Positives) ---")

real_errors = []

if os.path.exists(GOLD_DATA_PATH):
    for item in gold:
        extracted = extract_all(item['text'])
        
        for e in extracted:
            is_correct = (e['field_type'] == item['field_type'] and 
                          abs(e['start_char'] - item['start_char']) <= 3)
            
            if not is_correct:
                real_errors.append({
                    "Текст": item['text'][:50] + "...", 
                    "Витягнуто": f"{e['field_type']} ({e['span_text']})",
                    "Причина": "False Positive (невідповідність еталону)"
                })

if real_errors:
    df_err = pd.DataFrame(real_errors).head(10)
else:
    df_err = pd.DataFrame([["Дані збігаються", "Нічого", "Помилок не знайдено"]], 
                          columns=["Текст", "Витягнуто", "Причина"])

print(f"Знайдено реальних помилок: {len(real_errors)}")

print(f"Оновлення звіту: {AUDIT_REPORT_PATH}")
with open(AUDIT_REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write("# Audit Summary - Lab 4\n\n## Precision Metrics\n")
    f.write(report.to_markdown(index=False))
    f.write("\n\n## Real Error Analysis (Top 10 False Positives)\n")
    f.write(df_err.to_markdown(index=False))

Шукаю дані в: /Users/anastasiiakostyrka/Desktop/labs

--- Оцінка на Gold Standard ---
Field Type  Precision
    AMOUNT   0.928571
      DATE   0.800000
     PHONE   1.000000

--- Крок 3: Пошук реальних помилок (False Positives) ---
Знайдено реальних помилок: 4
Оновлення звіту: /Users/anastasiiakostyrka/Desktop/labs/docs/audit_summary_lab4.md
